In [ ]:
# ==========================
# 1 IMPORT LIBRARIES
# ==========================

import torch
import torch.nn as nn
import torch.optim as optim

import pandas as pd
from sklearn.model_selection import train_test_split

from torch.utils.data import TensorDataset, DataLoader


# ==========================
# 2 LOAD DATASET
# ==========================

data = pd.read_csv("data.csv")

X = data.drop("target", axis=1).values
y = data["target"].values


# ==========================
# 3 TRAIN TEST SPLIT
# ==========================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# ==========================
# 4 CONVERT TO TENSORS
# ==========================

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32).view(-1,1)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1,1)


# ==========================
# 5 CREATE DATASETS
# ==========================

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)


# ==========================
# 6 CREATE DATALOADERS
# ==========================

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)


# ==========================
# 7 BUILD MODEL
# ==========================

class ANNRegression(nn.Module):

    def __init__(self, input_size):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.model(x)


input_size = X_train.shape[1]

model = ANNRegression(input_size)


# ==========================
# 8 LOSS & OPTIMIZER
# ==========================

criterion = nn.MSELoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)


# ==========================
# 9 TRAIN MODEL
# ==========================

epochs = 50
best_loss = float("inf")

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        predictions = model(X_batch)

        loss = criterion(predictions, y_batch)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), "best_model.pth")


# ==========================
# 10 LOAD BEST MODEL
# ==========================

model.load_state_dict(torch.load("best_model.pth"))

model.eval()


# ==========================
# 11 EVALUATE MODEL
# ==========================

test_loss = 0

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        predictions = model(X_batch)

        loss = criterion(predictions, y_batch)

        test_loss += loss.item()

test_loss /= len(test_loader)

print("Test Loss:", test_loss)